# SingleTaskGP example

`robotorchan.models.SingleTaskGP` の基本的な使い方を、synthetic data を用いて確認します。

この Notebook では以下を扱います。

1. 学習データの作成
2. モデル構築
3. robotorchan 共通 API (`raw_*`, `supports_mll`, `make_mll()`)
4. GP の学習
5. posterior prediction
6. 可視化
7. `qLogExpectedImprovement` による次候補点の選択

> BoTorch / GPyTorch では数値安定性のため `torch.double` を推奨します。


In [ ]:
import matplotlib.pyplot as plt
import torch
from botorch.acquisition.logei import qLogExpectedImprovement
from botorch.fit import fit_gpytorch_mll
from botorch.optim import optimize_acqf

from robotorchan.models import SingleTaskGP

torch.set_default_dtype(torch.double)
torch.manual_seed(0)


## 1. Synthetic data

1 次元の連続変数 `x` に対する非線形関数を用います。


In [ ]:
def objective(x: torch.Tensor) -> torch.Tensor:
    return torch.sin(2 * torch.pi * x) + 0.2 * torch.cos(6 * torch.pi * x)

train_X = torch.linspace(0.05, 0.95, 10).unsqueeze(-1)
train_Y = objective(train_X)

print("train_X:", train_X.shape)
print("train_Y:", train_Y.shape)


## 2. Model construction

robotorchan の wrapper は BoTorch の `SingleTaskGP` の予測挙動を保ちつつ、
コンストラクタに渡した raw data を保持し、`make_mll()` を提供します。


In [ ]:
model = SingleTaskGP(train_X=train_X, train_Y=train_Y)

print(type(model).__name__)
print("supports_mll:", model.supports_mll)


## 3. robotorchan common API

`raw_*` はコンストラクタ入力のスナップショットです。
`condition_on_observations()` 等で更新された現在の学習状態ではありません。


In [ ]:
print("raw_data_names:", model.raw_data_names)
print("raw_train_X shape:", model.raw_train_X.shape)
print("raw_train_Y shape:", model.raw_train_Y.shape)
print("raw_train_Yvar:", model.raw_train_Yvar)
print("raw_data keys:", model.raw_data.keys())


## 4. Fit the model

Exact GP では `make_mll()` から marginal log likelihood を取得できます。


In [ ]:
mll = model.make_mll()
print(type(mll).__name__)

fit_gpytorch_mll(mll)
model.eval()


## 5. Posterior prediction


In [ ]:
test_X = torch.linspace(0.0, 1.0, 201).unsqueeze(-1)

with torch.no_grad():
    posterior = model.posterior(test_X)
    mean = posterior.mean.squeeze(-1)
    lower, upper = posterior.mvn.confidence_region()

lower = lower.squeeze(-1)
upper = upper.squeeze(-1)

print("posterior mean:", posterior.mean.shape)
print("posterior variance:", posterior.variance.shape)


## 6. Visualization


In [ ]:
plt.figure(figsize=(8, 4))
plt.scatter(train_X.squeeze(-1), train_Y.squeeze(-1), label="observed")
plt.plot(test_X.squeeze(-1), mean, label="posterior mean")
plt.fill_between(
    test_X.squeeze(-1),
    lower,
    upper,
    alpha=0.2,
    label="95% confidence interval",
)
plt.xlabel("x")
plt.ylabel("y")
plt.legend()
plt.show()


## 7. Bayesian optimization example

現在の最大観測値を `best_f` として `qLogExpectedImprovement` を構築し、
`[0, 1]` の範囲から次の候補点を 1 点選びます。


In [ ]:
best_f = train_Y.max()
acqf = qLogExpectedImprovement(model=model, best_f=best_f)

bounds = torch.tensor([[0.0], [1.0]])
candidate, acq_value = optimize_acqf(
    acq_function=acqf,
    bounds=bounds,
    q=1,
    num_restarts=5,
    raw_samples=64,
)

print("candidate:", candidate)
print("acquisition value:", acq_value)
print("objective(candidate):", objective(candidate))


## 8. When to use

`SingleTaskGP` は以下のような問題の基本モデルです。

- 単一タスクの回帰
- 主に連続変数からなる探索空間
- Exact GP が現実的なデータ数
- Bayesian Optimization のベースライン

カテゴリ変数を明示的に扱う場合は `MixedSingleTaskGP`、
高次元で少数の重要変数を仮定する場合は SAAS 系モデルも検討してください。
